# Direct Preference Alignment (DPO) with PyTorch FSDP on Amazon SageMaker Training jobs

## Lab 3 - LLM Inference

In this notebook, we are going to deploy the fine-tuned LLM using SageMaker Real-time endpoint

***

### Prerequistes

#### Setup and dependencies

In [ ]:
import boto3
from rich.pretty import pprint
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
model_id = "Qwen/Qwen3-4B-Instruct-2507"

model_name = "qwen3-4b-instruct-dpo"
endpoint_name = "qwen3-4b-instruct-dpo-endpoint"

***

### Test endpoint

In [ ]:
import boto3
import io
import json

In [ ]:
sagemaker_client = boto3.client(service_name="sagemaker-runtime")

### Iterator class for streaming inference

Utility class to parse streaming responses

In [ ]:
class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()

            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]

            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise

            if "PayloadPart" not in chunk:
                continue

            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

Utility function to parse model answer

In [ ]:
def parse_streaming_response(line_str):
    """Parse a single streaming response line and return (content, tool_call_delta)."""
    if not line_str.strip() or line_str.strip() == "data: [DONE]":
        return None, None

    if line_str.startswith("data: "):
        line_str = line_str[6:]

    try:
        data = json.loads(line_str)
        if "choices" in data:
            for choice in data["choices"]:
                delta = choice.get("delta", {})
                if "content" in delta and delta["content"]:
                    return delta["content"], None
                if "tool_calls" in delta:
                    return None, delta["tool_calls"]
    except json.JSONDecodeError:
        pass

    return None, None

In [ ]:
system = """
You are a helpful AI assistant that can answer questions and provide information.
You can use tools to help you with your tasks.
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate_bmi",
            "description": "Calculate BMI given weight in kg and height in meters",
            "parameters": {
                "type": "object",
                "properties": {
                    "weight_kg": {
                        "type": "number",
                        "description": "Property weight_kg",
                    },
                    "height_m": {"type": "number", "description": "Property height_m"},
                },
                "required": ["weight_kg", "height_m"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_weather",
            "description": 'Fetch weather information\n\nArgs:\nquery: The weather query (e.g., "weather in New York")\nnum_results: Number of results to return (default: 1)\n\nReturns:\nJSON string containing weather information\n',
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Property query"},
                    "num_results": {
                        "type": "integer",
                        "description": "Property num_results",
                    },
                },
                "required": ["query"],
            },
        },
    },
]

prompt = """
What is the weather in Rome, Italy?
"""

In [ ]:
request_body = {
    "messages": [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ],
    "tools": tools,
    "max_tokens": 4096,
    "temperature": 0.3,
    "top_p": 0.9,
    "stream": True,
}

response = sagemaker_client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    InferenceComponentName=ic_name,
    Body=json.dumps(request_body),
    ContentType="application/json",
)

generated_text = ""
tool_calls = []

for line in LineIterator(response["Body"]):
    if line:
        content, tool_call_delta = parse_streaming_response(line.decode("utf-8"))
        if content:
            generated_text += content
            print(content, end="", flush=True)
        if tool_call_delta:
            for tc in tool_call_delta:
                idx = tc.get("index", 0)
                while len(tool_calls) <= idx:
                    tool_calls.append({"type": "function", "function": {"name": "", "arguments": ""}})
                if "function" in tc:
                    if tc["function"].get("name"):
                        tool_calls[idx]["function"]["name"] = tc["function"]["name"]
                    if "arguments" in tc["function"]:
                        tool_calls[idx]["function"]["arguments"] += tc["function"]["arguments"]

if tool_calls:
    pprint(tool_calls)